# Filer — dev playground

**Development-only** notebook for poking at the low-level objects: configuration, file extraction/chunking, embeddings, the vector store + retrieval, suggestions, and LLMs.

Defaults to an **isolated scratch DB + LanceDB** (`notebooks/scratch/`, never your real app data) and an **offline LLM (`TestModel`)** — no API keys or network needed (the embedding model downloads once on first use). Run top-to-bottom.

Launch: `cd apps/backend && uv run jupyter lab`, then open this notebook.

## 1 · Setup — isolated scratch DB/LanceDB + migrations

In [ ]:
import shutil
from pathlib import Path

SCRATCH = True  # flip to False to use the real ~/Library/Application Support/Filer data

import filer_backend  # noqa: E402
import filer_backend.config as config
import filer_backend.storage.db as db
import filer_backend.storage.vectors as vectors

# Anchor scratch under apps/backend/notebooks/scratch regardless of launch cwd.
BASE = Path(filer_backend.__file__).resolve().parent.parent / "notebooks" / "scratch"


def _use_scratch():
    (BASE / "lancedb").mkdir(parents=True, exist_ok=True)
    config.db_url = lambda: f"sqlite:///{BASE / 'filer.db'}"
    db.db_url = config.db_url                 # storage.db imported the name
    db._engine = None                         # drop any cached engine
    db._SessionLocal = None
    config.lancedb_dir = lambda: BASE / "lancedb"
    vectors.lancedb_dir = config.lancedb_dir  # vectors imported the name
    vectors._db.cache_clear()


from filer_backend.migrations import run_migrations

if SCRATCH:
    _use_scratch()
run_migrations()
print("SCRATCH:", SCRATCH, "| db:", config.db_url(), "| lancedb:", config.lancedb_dir())

Helpers: reset the scratch sandbox, and build a tiny sample corpus to index.

In [ ]:
def reset_scratch():
    """Wipe the scratch sandbox and re-create empty tables."""
    if BASE.exists():
        shutil.rmtree(BASE)
    _use_scratch()
    run_migrations()
    print("scratch reset:", BASE)


def make_sample_corpus():
    root = BASE / "home"
    files = {
        "Finance/Invoices/inv_2026_q1.txt": "Fidelity Visa invoice statement, balance due, minimum payment, account 6665.",
        "Finance/2026/tax_return.txt": "Form 1040 federal income tax return, deductions and refund for 2026.",
        "Recipes/soup.txt": "Tomato basil soup recipe with garlic, olive oil and fresh herbs; simmer 30 minutes.",
        "Recipes/cookies.txt": "Chocolate chip cookie recipe: flour, butter, sugar, eggs; bake at 350F.",
    }
    for rel, text in files.items():
        p = root / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(text)
    return root

## 2 · Configuration

In [ ]:
from filer_backend.settings import get_settings, settings_for_profile

s = get_settings()
print("llm:", s.llm.provider, s.llm.model, "| embedding:", s.embedding.model)
print("hints:", repr(s.hints))
# Override via env before launch, e.g. FILER_LLM__PROVIDER=openai FILER_LLM__MODEL=gpt-4o-mini
# or define named profiles in config.toml and: settings_for_profile("openai")

## 3 · Files — extraction & chunking

In [ ]:
from filer_backend.indexing.extract import extract_text
from filer_backend.indexing.chunker import chunk_text

root = make_sample_corpus()
f = root / "Finance/Invoices/inv_2026_q1.txt"
text, parser = extract_text(f)
print("parser:", parser)
print("text:", text)
chunks = chunk_text(text)
print("chunks:", len(chunks))
chunks[:2]

## 4 · Embeddings

In [ ]:
import math
from filer_backend.embedding import get_embedder

emb = get_embedder()  # first call downloads bge-small (ONNX); cached afterwards
v = emb.embed(["bank statement balance due", "tomato basil soup recipe"])
print("dim:", emb.dim)


def cos(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb)


print("sim(statement, soup):", round(cos(v[0], v[1]), 3))

## 5 · Index the corpus → vector store → hybrid retrieval

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

from filer_backend.storage.db import get_session
from filer_backend.storage.models import IndexJob
from filer_backend.indexing.runner import run_index


def index_tree(rt):
    jid = uuid4().hex
    now = datetime.now(timezone.utc)
    s = get_session()
    try:
        s.add(IndexJob(id=jid, root_path=str(rt), status="pending", created_at=now, updated_at=now))
        s.commit()
    finally:
        s.close()
    return run_index(Path(rt), jid)


print("index:", index_tree(root))
print("chunks in LanceDB:", vectors.count())

In [ ]:
from filer_backend.filing.retrieval import hybrid_search, candidate_folders

query = "credit card statement payment due"
qvec = emb.embed([query])[0]
hits = hybrid_search(query, qvec, k=10)
for h in hits[:5]:
    print(round(h.get("_rrf", 0), 4), h["folder_path"].split("/")[-1], "/", h["filename"])

print("\ncandidate folders:")
for c in candidate_folders(hits):
    print("  ", round(c.score, 4), c.folder_path.split("/")[-1], c.files)

# Leave-one-out style: hide a file from its own results with exclude_file_ids={file_id}.

## 6 · Suggestions (offline `TestModel`)

In [ ]:
from filer_backend.storage.models import InboxFile
from filer_backend.filing.suggester import suggest_folders, folder_context, folder_history
from filer_backend.filing.llm import build_agent
from pydantic_ai.models.test import TestModel

inbox = InboxFile(
    id="demo", batch_id="nb",
    absolute_path=str(root / "Finance/Invoices/inv_2026_q1.txt"),
    filename="inv_2026_q1.txt", extension="txt", kind="document", size_bytes=80,
    modified_at=datetime.now(timezone.utc), status="ready", added_at=datetime.now(timezone.utc),
)

# A TestModel returns a fixed structured response — deterministic, offline.
agent = build_agent(get_settings())
fixed = TestModel(custom_output_args={"suggestions": [
    {"folder_path": str(root / "Finance/Invoices"), "confidence": 0.93, "rationale": "invoice", "is_new": False},
    {"folder_path": str(root / "Finance/2026/Healthcare"), "confidence": 0.4, "rationale": "infer new-year folder", "is_new": True},
]})
with agent.override(model=fixed):
    for sug in suggest_folders(inbox, agent=agent):
        print(round(sug.confidence, 2), "is_new=" + str(sug.is_new), sug.folder_path)

In [ ]:
# The context the suggestor assembles for the LLM:
s = get_session()
try:
    print("folder_context:", folder_context(s, [str(root / "Finance/Invoices")])[:8])
    print("folder_history:", folder_history(s))
finally:
    s.close()

## 7 · LLMs — schema & switching to a real provider

In [ ]:
import json
from filer_backend.filing.llm import SuggestionResponse, build_agent

print(json.dumps(SuggestionResponse.model_json_schema(), indent=2)[:600], "...")

# To call a REAL model instead of TestModel:
#   1) set a provider in env/config.toml (FILER_LLM__PROVIDER=openai, FILER_LLM__MODEL=gpt-4o-mini,
#      and the provider's API key) or run Ollama locally;
#   2) agent = build_agent(get_settings()); res = agent.run_sync("...your prompt..."); res.output

## 8 · Eval (optional) — leave-one-out on the scratch corpus

In [ ]:
import logging
logging.getLogger("filer_backend.filing.suggester").setLevel(logging.CRITICAL)  # quiet LLM fallback

from filer_backend.eval.core import run_eval

# No real LLM configured here, so the suggestor falls back to retrieval (a valid baseline).
res = run_eval(label="NB_DEMO", n=4, seed=1, out_dir=str(BASE / "evals"))
res["metrics"]["overall"]